# Part 4 of HW1
Author: Daniel Bond

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import roc_auc_score
import time, random

In [8]:
y = pd.read_csv("train_y_bronze.csv.gz")["TARGET"].values
test_ids = pd.read_csv("test_bronze.csv.gz", usecols=["SK_ID_CURR"])["SK_ID_CURR"]

oof = {"lgbm": np.load("oof_lgbm.npy"), "nn": np.load("oof_nn.npy")}
tst = {"lgbm": np.load("test_lgbm.npy"), "nn": np.load("test_nn.npy")}

for k in oof:
    print(k, roc_auc_score(y, oof[k]))
print("correlation:", np.corrcoef(oof["lgbm"], oof["nn"])[0, 1])

lgbm 0.7583658084891718
nn 0.7497056284249612
correlation: 0.9060653091846009


In [9]:
print("mean:", roc_auc_score(y, (oof["lgbm"] + oof["nn"]) / 2))

r = lambda a: rankdata(a) / len(a)
print("rank mean:", roc_auc_score(y, (r(oof["lgbm"]) + r(oof["nn"])) / 2))

for w in np.arange(0, 1.01, 0.1):
    print(f"w_lgbm={w:.1f}: {roc_auc_score(y, w * r(oof['lgbm']) + (1 - w) * r(oof['nn'])):.5f}")

mean: 0.7582046844413302
rank mean: 0.7582922549780857
w_lgbm=0.0: 0.74971
w_lgbm=0.1: 0.75208
w_lgbm=0.2: 0.75412
w_lgbm=0.3: 0.75585
w_lgbm=0.4: 0.75724
w_lgbm=0.5: 0.75829
w_lgbm=0.6: 0.75900
w_lgbm=0.7: 0.75936
w_lgbm=0.8: 0.75938
w_lgbm=0.9: 0.75905
w_lgbm=1.0: 0.75837


In [10]:
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
train_x = np.column_stack([logit(oof["lgbm"]), logit(oof["nn"])])
test_x = np.column_stack([logit(tst["lgbm"]), logit(tst["nn"])])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
meta = LogisticRegression()
oof_stack = cross_val_predict(meta, train_x, y, cv=skf, method="predict_proba")[:, 1]
print("CV AUC:", roc_auc_score(y, oof_stack))

meta.fit(train_x, y)
test_stack = meta.predict_proba(test_x)[:, 1]
pd.DataFrame({"SK_ID_CURR": test_ids.values, "TARGET": test_stack}).to_csv("sub_stack.csv", index=False)

CV AUC: 0.7593973525395669


### Results

Local: 0.759

Public leaderboard: 0.74806

Private leaderboard: 0.74534

Conclusion: Ensemble of the results of two previous submissions beat both leaderboards by ~0.001.